In [ ]:
import pandas as pd
import random
import csv
from tqdm import tqdm
import string

from collections import Counter

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ['hf_token']
)

In [65]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

In [66]:
pos_tags = []
pos_tag_story_begin = []
for story in df['story_raw']:
    doc = nlp(story)

    sent_list = list(doc.sents)

    for i, sent in enumerate(sent_list):
        first_word = sent[0]
        if i == 0:
            pos_tag_story_begin.append(first_word.pos_)

        pos_tags.append(first_word.pos_)
pos_counter = Counter(pos_tag_story_begin)

In [67]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(str(token))
        elif token.pos_ == 'VERB':
            verbs.add(str(token))
        elif token.pos_ == 'ADJ':
            adjectives.add(str(token))

In [68]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [69]:
def select_pos_tag(weighted=True):
    if weighted:
        tags = list(pos_counter.keys())
        frequencies = list(pos_counter.values())
        sample = random.choices(tags, weights=frequencies, k=1)[0]
    else:
        sample = pos_tags[random.randint(0, len(pos_tags) - 1)]
    
    return sample

In [70]:
story_features = [
    "dialoog",
    "in medias res",
    "een morele les",
    "een onverwachte wending",
    "een onbetrouwbare verteller",
    "vooruitwijzing",
    "ironie",
    "innerlijke monoloog",
    "symboliek",
    "een MacGuffin",
    "een niet-lineaire tijdlijn",
    "een omgekeerde tijdlijn",
    "circulaire verhaalsstructuur",
    "een flashback",
    "een geneste structuur",
    "een dwaalspoor",
    "meerdere perspectieven",
    "de vierde wand",
    "een cliffhanger",
    "een antiheld",
    "contrast (juxtapositie)",
    "climax-structuur"
]

verhaalelementen = [
    "sprekende dieren",
    "fantasiewerelden",
    "tijdreizen",
    "een deadline of tijdslimiet",
    "ruimteverkenning",
    "mystieke wezens",
    "onderwateravonturen",
    "dinosaurussen",
    "piraten",
    "superhelden",
    "sprookjes",
    "het heelal",
    "verborgen schatten",
    "magische landen",
    "betoverde bossen",
    "geheime genootschappen",
    "robots en technologie",
    "sport",
    "schoolleven",
    "vakanties",
    "culturele tradities",
    "magische voorwerpen",
    "verloren beschavingen",
    "ondergrondse werelden",
    "vervlogen tijdperken",
    "onzichtbaarheid",
    "reusachtige wezens",
    "miniatuurwerelden",
    "ontmoetingen met buitenaardse wezens",
    "behekste plekken",
    "vormverandering",
    "eilandavonturen",
    "ongewone voertuigen",
    "geheime missies",
    "droomwerelden",
    "virtuele werelden",
    "raadsels",
    "rivaliteit tussen broers en zussen",
    "schattenjachten",
    "sneeuwavonturen",
    "seizoenswisselingen",
    "mysterieuze kaarten",
    "koninkrijken",
    "levende objecten",
    "tuinen",
    "verloren steden",
    "de kunsten",
    "de hemel"
]

"""

"""

'\n\n'

In [71]:
# Rags to Riches Riches to Rags Man in a Hole Double Man in a Hole Icarus Cinderella Oedipus (macro niveau)

# TP1 - Opportunity TP2 - Change of Plans TP3 - Point of No Return TP4 - Major Setback TP5 - Climax The introductory event that sets the stage for the narrative.
# A pivotal moment where the main goal of the narrative is defined or altered.
# The commitment point beyond which the protagonists are invested in goals.
# A critical juncture where the protagonists face significant challenges or failures.
# The peak of the narrative arc, encompassing the resolution of the central conflict.

In [72]:
def generate_prompt():
    chosen_noun = random.choice(nouns)
    chosen_adjective = random.choice(adjectives)
    chosen_verb = random.choice(verbs)
    chosen_pos_tag = select_pos_tag()
    chosen_letter = random.choice(string.ascii_lowercase)
    chosen_feature = random.choice(story_features)
    element = random.choice(verhaalelementen)

    prompt = f"""Vertel een verhaal. Het verhaal moet het volgende werkwoord bevatten: {chosen_verb}, het volgende zelfstandig naamwoord {chosen_noun} en het volgende bijvoegelijk naamwoord {chosen_adjective}.
    Het verhaal moet het volgende kenmerk bevatten: {chosen_feature} en het volgende verhaal element: {element}.
Begin het verhaal met een {chosen_pos_tag}."""

    return prompt

In [73]:
for _ in tqdm(range(5)):
    prompt = generate_prompt()
    print("prompt:", prompt)
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b:cerebras",
        messages=[
            {"role": "system", "content": f"""
                Je bent een verteller van een kort verhaal (100-600 woorden).
                Je bent een kind tussen de 4 en 12 en je vertelt een verhaal aan klasgenoten. Gebruik woorden en taalconstructies die kinderen van die leeftijd gebruiken. 
                Ik herhaal: gebruik makkelijke zinsconstructies.
                Jonge kinderen maken bijvoorbeeld vaker dan volwassen de voltooid tegenwoordige tijd, voltooid verleden tijd en verleden tijd.
                Geef het verhaal geen titel of introductie. Het verhaal hoeft geen ego-narratie te zijn, mensen gebruiken een verhaal zelden om hun eigen perspectief te vertellen. Het mag dus vertelt worden
                vanuit het perspectief van iemand anders.
                """},
            {"role": "user", "content": prompt},
        ],
    )

    completion = response.choices[0].message.content.strip()
    print(completion)

  0%|          | 0/5 [00:00<?, ?it/s]

prompt: Vertel een verhaal. Het verhaal moet het volgende werkwoord bevatten: uitmaken, het volgende zelfstandig naamwoord ochtends en het volgende bijvoegelijk naamwoord nieuw.
    Het verhaal moet het volgende kenmerk bevatten: een dwaalspoor en het volgende verhaal element: raadsels.
Begin het verhaal met een DET.


 20%|██        | 1/5 [00:00<00:03,  1.08it/s]

De kleine klas ging op een speurtocht in het bos. We kregen een kaart met raadsels en een beetje een dwaalspoor.  

Ochtends hadden we allemaal een rugzak vol koekjes en een nieuw vergrootglasmutsje van juf. Ze vertelde ons dat we een verborgen schat moesten uitzoeken. Het eerste raadsel was: “Wat is wit, loopt niet en maakt je blij?” We dachten even en riepen: “Sneeuw!” Maar de kaart liet zien dat de sneeuw geen spoor had. Dat was een dwaalspoor!  

We volgden de pijl naar een oude eik. Daar lag een brief die zei: “Om de volgende aanwijzing te vinden, moet je het geluid van de wind uitmaken.” Ik zei tegen de wind: “Wind, maak het geluid stil!” En ineens hoorden we een fluisterende klank tussen de takken.  

De geluiden leidden ons naar een holle boomstam. In de stam zat een klein, glinsterend doosje. Het doosje was nieuw en glansde als een ster. “Wat zit erin?” vroeg Sam. We openden het en zagen een kaart met een nieuw raadsel: “Ik ben een sleutel, maar ik sluit niets. Ik maak je ster

 40%|████      | 2/5 [00:01<00:02,  1.08it/s]

...

Plotseling hoorde ik een hard geluid. Het was de wildwaterbaan in het pretpark die me ineens overviel, net toen ik met mijn familie op vakantie was. We waren net uit de auto gestapt, en ik had mijn zonnebril op, want het was super heet. 

Eerst ging ik met mijn kleine broertje op de torens van de speelplaats. Daarna sprong ik naar de glijbaan, want ik vond het wel een beetje gek dat we zo snel naar beneden gingen. Mijn papa riep: “Kijk uit!” en ik viel bijna, maar ik hield me vast aan de rand. 

Daarna, een paar dagen later, toen we terug naar huis gingen, begon mama te vertellen over de reis die we eerder hadden gemaakt. Ze zei: “We gingen eerst naar de bergen, dan naar het strand, en daarna naar het pretpark.” Ik dacht even: “Wacht, was dat niet eerst?” En ik zag een foto van de wildwaterbaan in ons vakantieboek. Het leek wel of de tijd een beetje rondjes draaide. 

Even later, in de zomer, mocht ik met mijn vrienden naar de nieuwe wildwaterbaan gaan. We waren allemaal gek van het wa

 80%|████████  | 4/5 [00:03<00:00,  1.22it/s]

Snel waren we klaar voor de zomervakantie. Mama zei: “We gaan naar het vliegveld morgen, dan vliegen we naar het strand.” Ik sprong van blijdschap en schreeuwde: “Ik kan niet wachten!”  

De avond voor de reis legde papa een grote koffer op het bed. Hij legde ook een klein rood kaartje in mijn hand. “Dit is ons ticket,” fluisterde hij, “en het is heel speciaal, want het wordt onzichtbaar als je er niet naar kijkt.”  

De volgende ochtend stonden we vroeg op. De bus reed hard naar het vliegveld. In de bus zat mijn beste vriend Tim. Hij vroeg: “Heb jij je zonnebril al meegenomen?” Ik antwoordde: “Ja, ik heb hem al op mijn hoofd gelegd.”  

Op het vliegveld zagen we veel grote vliegtuigen. Een stewardess kwam naar ons toe en zei: “Willen jullie nog een drankje?” Ik zei: “Ja, alsjeblieft, een sapje.”  

Toen we in het vliegtuig zaten, vertelde mama een verhaaltje over een onzichtbare piratenboot die over de wolken zeilde. “Dat is gewoon een spelletje,” zei ze lachend.  

We landden op een 

100%|██████████| 5/5 [00:04<00:00,  1.12it/s]

Ik ben met de klas op een bootje naar een klein eilandje gegaan. De juf had ons beloofd dat we een echte schattenjacht zouden doen. We hadden een grote kaart en een kompas, en iedereen riep blij: “Ja!”  

We stapten op het zand en volgden de dikke, rode lijn op de kaart. Eerst vond ik een grote eik, dan een krab die op de rotsen kroop. Maar toen kwam er een dwaalspoor. De weg eindigde bij een dode boom en er was geen schatkist te zien. “O nee,” zei Tom, “we lopen in de mindere kant van de jungle.”  

De juf zei dat we even terug moesten gaan en een andere route moesten zoeken. We draaiden om en keken nog eens goed naar de kaart. Daar stond een blauwe streep die naar een geheim grotje leidde. We sprongen over een klein beekje en klommen over een stapel rotsen.  

In de grot vonden we een oude kist met gouden munten en een glinsterende parel. Iedereen juichte en klapte. De juf gaf ons een stukje van de schat als beloning. Ze zei dat we het aan huis mochten meenemen, maar alleen als we he